# 03 - Extract Ember Yearly Electricity Data

## Purpose
Filter and reshape the Ember yearly electricity dataset to extract three
variable types relevant to the CBAM analysis: grid carbon intensity,
installed capacity by fuel type, and electricity generation by fuel type.

## Input
`data/raw/yearly_full_release_long_format.csv`

## Outputs
- `data/processed/ember_co2_intensity.csv`
- `data/processed/ember_capacity.csv`
- `data/processed/ember_generation.csv`

## Notes
- Source file is in long format with many variables stacked.
  Extraction filters to relevant rows only before reshaping.
- Area type includes countries, regions and aggregates. Filter to
  "Country or economy" only to avoid double counting.
- Year range: 2000 onwards. For CBAM purposes most recent year
  is primary, but full time series retained for trend analysis.

In [1]:
import pandas as pd
from pathlib import Path

raw_path = Path("/Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/raw/yearly_full_release_long_format.csv")

df = pd.read_csv(raw_path)

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nArea types: {df['Area type'].unique()}")
print(f"\nYear range: {df['Year'].min()} to {df['Year'].max()}")
print(f"\nUnique categories: {df['Category'].unique()}")

Shape: (370961, 18)

Columns: ['Area', 'ISO 3 code', 'Year', 'Area type', 'Continent', 'Ember region', 'EU', 'OECD', 'G20', 'G7', 'ASEAN', 'Category', 'Subcategory', 'Variable', 'Unit', 'Value', 'YoY absolute change', 'YoY % change']

Area types: <StringArray>
['Country or economy', 'Region']
Length: 2, dtype: str

Year range: 2000 to 2025

Unique categories: <StringArray>
[              'Capacity',     'Electricity demand', 'Electricity generation',
    'Electricity imports', 'Power sector emissions']
Length: 5, dtype: str


In [ ]:
# Check year coverage - how many countries have data per year
countries_only = df[df["Area type"] == "Country or economy"]

# For CO2 intensity specifically, check country count per year
intensity_rows = countries_only[
    (countries_only["Category"] == "Power sector emissions") &
    (countries_only["Subcategory"] == "CO2 intensity")
]

coverage = intensity_rows.groupby("Year")["Area"].nunique()
print("Countries with CO2 intensity data per year:")
print(coverage.tail(10))  # Last 10 years

Countries with CO2 intensity data per year:
Year
2016    214
2017    214
2018    214
2019    214
2020    213
2021    213
2022    213
2023    212
2024    196
2025     90
Name: Area, dtype: int64


In [4]:
# Filter to countries only and relevant variable types
countries_only = df[df["Area type"] == "Country or economy"].copy()

# Aggregate fuel: keep only Clean and Fossil summary rows, drop intermediate groupings
intermediate_aggregates = [
    "Gas and Other Fossil",
    "Hydro, Bioenergy and Other Renewables", 
    "Wind and Solar"
]

# Define filters for each of the three variable types
mask_intensity = (
    (countries_only["Category"] == "Power sector emissions") &
    (countries_only["Subcategory"] == "CO2 intensity")
)

mask_capacity = (
    (countries_only["Category"] == "Capacity") &
    (
        (countries_only["Subcategory"] == "Fuel") |
        (
            (countries_only["Subcategory"] == "Aggregate fuel") &
            (~countries_only["Variable"].isin(intermediate_aggregates))
        )
    )
)

mask_generation = (
    (countries_only["Category"] == "Electricity generation") &
    (
        (countries_only["Subcategory"] == "Fuel") |
        (
            (countries_only["Subcategory"] == "Aggregate fuel") &
            (~countries_only["Variable"].isin(intermediate_aggregates))
        )
    )
)

cols_to_drop = ["Area type", "EU", "OECD", "G20", "G7", "ASEAN",
                "YoY absolute change", "YoY % change"]

df_filtered = countries_only[mask_intensity | mask_capacity | mask_generation].copy()
df_filtered = df_filtered.drop(columns=cols_to_drop)

print(f"Original shape: {df.shape}")
print(f"Filtered shape: {df_filtered.shape}")
print(f"\nVariables retained:")
print(df_filtered.groupby(["Category", "Subcategory", "Variable"]).size().to_string())

Original shape: (370961, 18)
Filtered shape: (193936, 10)

Variables retained:
Category                Subcategory     Variable        
Capacity                Aggregate fuel  Clean                5322
                                        Fossil               5322
                                        Renewables           5322
                        Fuel            Bioenergy            5281
                                        Coal                 5213
                                        Gas                  5142
                                        Hydro                5228
                                        Nuclear              4904
                                        Other Fossil         5321
                                        Other Renewables     4676
                                        Solar                5304
                                        Wind                 5106
Electricity generation  Aggregate fuel  Clean               10826
      

In [5]:
# Check units per variable to make sure nothing unexpected
print(df_filtered.groupby(["Category", "Variable", "Unit"])["Value"].count().to_string())

Category                Variable          Unit    
Capacity                Bioenergy         GW          3548
                        Clean             GW          5211
                        Coal              GW          2155
                        Fossil            GW          5249
                        Gas               GW          3098
                        Hydro             GW          3986
                        Nuclear           GW           849
                        Other Fossil      GW          4356
                        Other Renewables  GW          1345
                        Renewables        GW          5211
                        Solar             GW          5146
                        Wind              GW          3591
Electricity generation  Bioenergy         %           5346
                                          TWh         5372
                        Clean             %           5413
                                          TWh         5413
     

## Output
`data/processed/ember_electricity.csv`

Single file retaining Category column to distinguish between:
- Capacity (GW by fuel type)
- Electricity generation (TWh and % share by fuel type)
- Power sector emissions (CO2 intensity in gCO2/kWh)

In [8]:
output_path = Path("/Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/processed/country_grid_electricity.csv")
df_filtered.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Final shape: {df_filtered.shape}")

Saved to: /Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/processed/country_grid_electricity.csv
Final shape: (193936, 10)
